# **CMU 16825 Learning for 3D Vision – Final Project**
## **Uncertainty-Aware Hybrid Rendering with Gaussian Splatting and NeRF for High-Fidelity Synthesis**
### Team 25 Patrick Chen *(Andrew ID: bochunc)*

---

# Abstract

Efficient 3D rendering often involves a trade-off between speed and quality. **3D Gaussian Splatting (GS)** offers fast rendering but lacks fine detail, while **NeRF** delivers high fidelity at a high computational cost. We propose an **uncertainty-aware hybrid rendering framework** that combines the strengths of both, selectively applying NeRF to uncertain regions identified via a lightweight U-Net or MLP predictor trained on GS-derived features. Our best-performing model, **Hybrid-MLP 50%**, achieves the highest SSIM score (**0.892**) and a PSNR of **24.75**, outperforming GS-only (SSIM: 0.819, PSNR: 22.23) and matching NeRF-only (SSIM: 0.891, PSNR: 24.76), while reducing the total rendering time to **0.668s**—faster than NeRF-only (**0.827s**) and the state-of-the-art SS Mip-NeRF (**0.864s**). Meanwhile, the **Hybrid-UNet 50%** model offers similarly high quality (SSIM: **0.892**, PSNR: **24.73**) with a slightly higher runtime of **0.673s**, showcasing the robustness of our framework across model choices. These results demonstrate that our hybrid approach delivers superior perceptual fidelity with reduced computational cost by leveraging NeRF only where needed.


---

# Table of Contents
- [Introduction](#introduction)
- [Related Work](#related-work)
- [Method](#method)
- [Neural Uncertainty Feature](#neural-uncertainty-feature)
- [Qualitative Result](#qualitative-result)
- [Quantitative Result](#quantitative-result)
- [References](#references)

---

# Introduction

Rendering 3D scenes typically trades off between efficiency and visual quality. **Gaussian Splatting (GS)** achieves fast rendering speeds but tends to blur fine textures. On the other hand, **Neural Radiance Fields (NeRF)** provide photorealistic quality but are computationally expensive for high-resolution scenes. We aim to develop a **hybrid rendering pipeline** that selectively applies NeRF rendering only to visually uncertain regions, identified through **uncertainty prediction**. This approach combines the advantages of both methods: fast rendering speeds and high-fidelity output, overcoming the limitations faced by either method individually.

---

# Related Work

- **NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis [1]**  
  Introduced novel view synthesis with high-quality results, but suffers from slow inference speed.

- **3D Gaussian Splatting for Real-Time Radiance Field Rendering [2]**  
  Enabled fast 3D scene rendering, but lost high-frequency texture details compared to NeRF.

- **SS Mip-NeRF: Supersampled Mip-NeRF [3]**  
  Enhanced Mip-NeRF’s detail preservation by combining cone-based rendering with supersampling for superior anti-aliasing.

Our project builds upon these works by integrating uncertainty estimation into hybrid rendering for improved trade-off.

---

# Method


![Uncertainty-Aware Hybrid Rendering Framework](./Figures/Proposed_Uncertainty_Framework.png)

We propose an uncertainty-aware hybrid rendering framework that leverages the speed of 3D Gaussian Splatting (GS) and the fidelity of NeRF through uncertainty-guided pixel-wise refinement. As shown above figure, the framework consists of three stages:

**Training Phase:**

**(1) 3D GS Training:**  
A GS model is trained on multi-view RGB images and camera poses to enable efficient rendering and produce auxiliary features for uncertainty prediction. The training minimizes a perceptual loss that combines L1 and SSIM losses:

$$
\mathcal{L}_{\text{GS}} = \|I_{\text{pred}} - I_{\text{gt}}\|_1 + \lambda_{\text{SSIM}} \cdot (1 - \text{SSIM}(I_{\text{pred}}, I_{\text{gt}})),
$$

where $\lambda_{\text{SSIM}}$ is a learnable weight and SSIM is computed over RGB channels.

---

**(2) NeRF Training:**  
A NeRF model is trained independently using the same inputs to generate high-fidelity volume renderings. The loss combines coarse and fine-level MSE losses:

$$
\mathcal{L}_{\text{NeRF}} =
\begin{cases}
\text{MSE}(I_{\text{coarse}}, I_{\text{gt}}), & \text{if epoch} < 100 \\
\alpha \cdot \text{MSE}(I_{\text{coarse}}, I_{\text{gt}}) + (1 - \alpha) \cdot \text{MSE}(I_{\text{fine}}, I_{\text{gt}}), & \text{otherwise}
\end{cases}
$$

with

$$
\alpha = \max\left(0.9 - \frac{\text{epoch} - 100}{\text{num\_epochs} - 100}, 0.5\right)
$$

to gradually introduce fine supervision.

---

**(3) Uncertainty Model Training:**  
A lightweight U-Net or MLP is trained to predict per-pixel SSIM error from GS-rendered outputs. It uses four GS-derived features (alpha sum, color variance, view direction, 2D covariance area) and camera poses to produce a per-pixel uncertainty map. The training minimizes the smooth L1 (Huber) loss:

$$
\mathcal{L}_{\text{uncertainty}} = \text{Huber}(u_{\text{pred}}, u_{\text{target}}),
$$

where $u_{\text{target}}$ is the SSIM error map and $u_{\text{pred}}$ is the model's prediction.

---

**Inference Phase:**
- **Input during Inference**:  
  - A new camera pose is provided to the pipeline.

- **Steps during Inference**:

  (1). **GS Rendering**:  
  - The GS model renders an initial full image and extracts GS feature maps.

  (2). **Uncertainty Prediction**:  
  - The uncertainty model takes the GS feature maps and predicts an uncertainty score for each pixel.

  (3). **Pixel Selection**:  
  - The top-k% most uncertain pixels are selected for refinement.

  (4). **NeRF Refinement**:  
  - The NeRF model re-renders only the selected uncertain pixels based on the input camera pose.

  (5). **Merging**:  
  - The final image is composed by combining GS-rendered pixels with NeRF-refined pixels at uncertain regions.

This hybrid approach allows the system to preserve the efficiency of GS while selectively enhancing critical regions using NeRF, achieving a balance between fast rendering and high visual quality.

---

# Neural Uncertainty Feature


![Feature Extraction for Neural Unceratainty Prediction](./Figures/Feature_Extraction.png)

We extract four types of features from the GS model for uncertainty prediction. The input to the feature extraction consists of the output from the trained 3D Gaussian Splatting (GS) model, which includes the parameters of each 3D splat: 3D mean position, orientation quaternion, opacity (alpha), scales (radii), and RGB color values. These attributes are processed together with the known camera pose to generate per-pixel feature maps over the rendered image plane. To be more specific, a 2D Gaussian is represented by the following expression:

$$
f(x; \mu_i, \Sigma_i) = \frac{1}{2\pi \sqrt{|\Sigma_i|}} \exp\left( -\frac{1}{2} (x-\mu_i)^T \Sigma_i^{-1} (x-\mu_i) \right)
$$

where:
- $x$ is a 2D vector that represents the pixel location
- $\mu_i$ is the 2D vector representing the mean of the $i$-th 2D Gaussian
- $\Sigma_i$ is the covariance of the 2D Gaussian

Given the opacity $o_i$ of a 3D Gaussian, the alpha value at pixel $x$ is computed as:

$$
\alpha_i(x) = o_i \exp\left(P(x,i)\right)
$$

where

$$
P(x,i) = -\frac{1}{2} (x-\mu_i)^T \Sigma_i^{-1} (x-\mu_i)
$$

Thus, $\alpha_i(x)$ represents the contribution of the $i$-th Gaussian to the pixel opacity at location $x$. 

We extract the four feature maps as the following for training our uncertainty prediction model:


- **Alpha sum**: Measures the total accumulated opacity at each pixel.

  $$
  \text{Alpha Sum}(x, y) = \sum_{i=1}^{N} \alpha_i(x, y)
  $$

  The alpha sum indicates the overall visibility along the viewing ray at a given pixel. Pixels with low accumulated opacity (e.g., transparent or sparsely covered regions) often suffer from high rendering uncertainty due to insufficient splat contributions. Conversely, very dense opacity could cause over-blending and loss of sharpness. Thus, alpha sum provides a strong signal for identifying uncertain regions.


- **Color variance**: Reflects the RGB variance of splat colors weighted by transmittance and opacity.

  $$
  \text{Color Variance}(x, y) = \frac{1}{3} \sum_{c \in \{R,G,B\}} \left( \frac{ \sum_{i=1}^{N} w_i(x,y) (c_i(x,y) - \bar{c}(x,y))^2 }{ \sum_{i=1}^{N} w_i(x,y) } \right)
  $$

  where

  $$
  w_i(x, y) = \alpha_i(x, y) \times \text{transmittance}_i(x, y),
  $$

  and

  $$
  \bar{c}(x,y) = \frac{\sum_{i=1}^{N} w_i(x,y) \, c_i(x,y)}{\sum_{i=1}^{N} w_i(x,y)}
  $$

  Here, $\bar{c}(x,y)$ represents the weighted average color at pixel $(x,y)$, computed using the weights from opacity and transmittance. It serves as the local mean color, allowing us to measure how much each splat's color deviates from the expected average appearance at that pixel. A high deviation indicates the presence of multiple surfaces, lighting variations, or texture discontinuities, leading to greater uncertainty. Therefore, regions with large color variance are more likely to suffer from rendering errors and benefit from NeRF-based refinement.
  


- **2D covariance area**: Captures the average projected footprint size (area of splat spread).

  $$
  \text{2D Covariance Area}(x, y) = \frac{ \sum_{i=1}^{N} \alpha_i(x, y) \det(\Sigma_i) }{ \sum_{i=1}^{N} \alpha_i(x, y) }
  $$

  where $\Sigma_i$ is the 2D covariance matrix of the $i$-th splat.

  The 2D covariance area measures how spread out the projected splats are on the image plane. Larger footprint areas imply a higher degree of depth uncertainty, motion blur, or blending between objects. Pixels associated with large projected areas tend to be more uncertain due to ambiguities in fine geometry or appearance, making this feature a reliable predictor of error-prone regions.


- **View direction cosine**: Represents the weighted cosine of the viewing angle.

  $$
  \text{View Direction Cosine}(x, y) = \frac{ \sum_{i=1}^{N} \alpha_i(x, y) \cos(\theta_i) }{ \sum_{i=1}^{N} \alpha_i(x, y) }
  $$

  where $\theta_i$ is the angle between the camera viewing direction and the splat’s surface orientation.

  The view direction cosine quantifies how oblique the viewing angle is relative to the splat’s normal. Pixels viewed at grazing angles often exhibit foreshortening, lower surface visibility, and higher rendering distortion. As such, highly oblique view angles are strongly correlated with greater rendering uncertainty, making this an important geometric feature for uncertainty prediction.


These four feature maps, each of size $[H, W]$, totally $[H, W, 4]$, are fed into a neural network (MLP or UNet) to predict per-pixel SSIM-based uncertainty.

---

# Qualitative Results

### Dataset Introduction

We use the NeRF Synthetic Dataset, which consists of photorealistic scenes rendered from synthetic 3D assets. The dataset provides multi-view RGB images along with ground-truth camera poses for each scene. Both the training and validation sets contain 100 images and their corresponding camera poses, while the test set contains 200 images and camera poses. In our experiments, we focus on the "materials" scene.

In our framework, the input includes multi-view RGB images and their corresponding camera poses. The expected output is a high-quality rendered novel view, reconstructed using a hybrid rendering pipeline that combines 3D Gaussian Splatting and uncertainty-guided pixel-wise NeRF refinement.

For the following results, we compare two models for uncertainty prediction: a 110K-parameter U-Net achieving a Spearman correlation of 0.890, and a lightweight 1K-parameter two-layer MLP achieving a Spearman correlation of 0.756.

### Implemented Uncertainty Prediction Model: U-Net

The following table presents the hybrid rendering results using U-Net with 110K parameters for uncertainty prediction, where the top \( k\% \) of pixels with the highest uncertainty are re-rendered using pixel-wise NeRF refinement.


| Uncertainty Models | Camera View Index | Top k% Uncertainty Threshold | Ground-Truth Image | GS Prediction Image | NeRF Prediction Image | SSIM Error Map | Predicted Uncertainty | Mask After Top k% Thresholding | Hybrid Rendering Result |
|:-------------------:|:-----------------:|:----------------------------:|:------------------:|:-------------------:|:----------------------:|:-------------:|:----------------------:|:-----------------------------:|:-----------------------:|
| U-Net | 0 | k = 5 | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_tunet2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| U-Net | 0 | k = 10 | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_tunet2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| U-Net | 0 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| U-Net | 0 | k = 50 | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_tunet2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |

We observe that when \(k = 30\) or \(k = 50\), the hybrid rendering achieves fidelity comparable to full NeRF rendering. As k increases, more uncertain regions are re-rendered by NeRF, leading to improved perceptual quality. When \(k = 5\) or \(k = 10\), only a small fraction of pixels are refined, resulting in noticeable quality gaps compared to ground-truth images. However, at \(k = 30\) and \(k = 50\), the majority of important visual structures are corrected, while the background and simple areas remain efficiently rendered by Gaussian Splatting. This demonstrates that uncertainty-guided hybrid rendering can effectively balance fidelity and efficiency by selectively applying NeRF refinement only where necessary.


The following table shows the 360 degree reconstruction scene using the hybrid rendering with the proposed U-Net and top k% uncertained pixels selection:


| Uncertainty Models | Top k% Uncertainty Threshold  | GS Prediction GIF | NeRF Prediction GIF | Hybrid Rendering GIF |
|:-------------------:|:----------------------------:|:-----------------:|:-------------------:|:--------------------:|
| U-Net               | k = 5                        | ![1](./output_hybrid_threshold0.05_renderall_init15000_tunet2/val_testgs.gif) | ![2](./output_hybrid_threshold0.05_renderall_init15000_tunet2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.05_renderall_init15000_tunet2/val_testhybrid.gif) |
| U-Net               | k = 10                        | ![1](./output_hybrid_threshold0.1_renderall_init15000_tunet2/val_testgs.gif) | ![2](./output_hybrid_threshold0.1_renderall_init15000_tunet2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.1_renderall_init15000_tunet2/val_testhybrid.gif) |
| U-Net               | k = 30                        | ![1](./output_hybrid_threshold0.3_renderall_init15000_tunet2/val_testgs.gif) | ![2](./output_hybrid_threshold0.3_renderall_init15000_tunet2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.3_renderall_init15000_tunet2/val_testhybrid.gif) |
| U-Net               | k = 50                        | ![1](./output_hybrid_threshold0.5_renderall_init15000_tunet2/val_testgs.gif) | ![2](./output_hybrid_threshold0.5_renderall_init15000_tunet2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.5_renderall_init15000_tunet2/val_testhybrid.gif) |

From the table above, we observe that when \( k = 30 \) and \( 50 \), the 360-degree reconstruction remains nearly identical to the full NeRF rendering. The hybrid results achieve comparable fidelity, while most of the pixels are still rendered by Gaussian Splatting. This demonstrates that only 30\% or 50\% of pixels need to be refined by NeRF to maintain high perceptual quality.

### Implemented Uncertainty Prediction Model: MLP
The following table changes the uncertainty prediction model from 110K-parameter U-Net to a simple two-layer MLP with only 1K parameters. We can check whether the complexity of the uncertainty prediction model is necessary for hybrid rendering fidelity or not.


| Uncertainty Models | Camera View Index | Top k% Uncertainty Threshold | Ground-Truth Image | GS Prediction Image | NeRF Prediction Image | SSIM Error Map | Predicted Uncertainty | Mask After Top k% Thresholding | Hybrid Rendering Result |
|:-------------------:|:-----------------:|:----------------------------:|:------------------:|:-------------------:|:----------------------:|:-------------:|:----------------------:|:-----------------------------:|:-----------------------:|
| MLP | 0 | k = 5 | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| MLP | 0 | k = 10 | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| MLP | 0 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |
| MLP | 0 | k = 50 | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gt.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/cam_view_val_index_val000/val000_hybrid_render.png" width="250"> |


From the table above, we observe the hybrid rendering results using an MLP-based uncertainty predictor across different top \( k\% \) thresholds. When \(k = 5\) and \( 10 \), only a small portion of uncertain regions are re-rendered by NeRF, leading to noticeable fidelity gaps compared to the ground-truth. However, when \(k = 30\) and \(k = 50\), the hybrid renderings achieve perceptual quality that is nearly indistinguishable from full NeRF predictions. This demonstrates that re-rendering only 30\% to 50\% of the pixels based on uncertainty estimation is sufficient to recover high-fidelity details, while the majority of the scene is still efficiently rendered by 3D Gaussian Splatting. The result is very similar to the one using U-Net as the uncertainty prediction model, indicating that simpler model can achieve comparable results in this framework.

The following table shows the 360 degree reconstruction scene using the hybrid rendering with the proposed MLP and top k% uncertained pixels selection:

| Uncertainty Models | Top k% Uncertainty Threshold  | GS Prediction GIF | NeRF Prediction GIF | Hybrid Rendering GIF |
|:-------------------:|:----------------------------:|:-----------------:|:-------------------:|:--------------------:|
| MLP               | k = 5                        | ![1](./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/val_testgs.gif) | ![2](./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.05_renderall_init15000_simple-mlp2/val_testhybrid.gif) |
| MLP               | k = 10                        | ![1](./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/val_testgs.gif) | ![2](./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.1_renderall_init15000_simple-mlp2/val_testhybrid.gif) |
| MLP               | k = 30                        | ![1](./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/val_testgs.gif) | ![2](./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/val_testhybrid.gif) |
| MLP               | k = 50                        | ![1](./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/val_testgs.gif) | ![2](./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/val_testnerf.gif) | ![3](./output_hybrid_threshold0.5_renderall_init15000_simple-mlp2/val_testhybrid.gif) |

The results above are very similar to those generated by the U-Net model. When \(k = 30\) or \(k = 50\), the 360-degree reconstruction remains nearly identical to full NeRF rendering. The hybrid method achieves comparable fidelity, while the majority of pixels are still rendered by Gaussian Splatting. This demonstrates that refining only 30\% or 50\% of pixels with NeRF is sufficient to maintain high perceptual quality. Moreover, it shows that the uncertainty prediction model does not need to be complex, as even a lightweight MLP can achieve strong fidelity.


### More Rendering Results Using These Two Models Under Different Camera Views

As k = 30 could provide high fidelity resutls, we further view the hybrid rendering results with k = 30 under different camera views (by changing the camera view index in the test dataset). The following table shows the results.

| Uncertainty Models | Camera View Index | Top k% Uncertainty Threshold | Ground-Truth Image | GS Prediction Image | NeRF Prediction Image | SSIM Error Map | Predicted Uncertainty | Mask After Top k% Thresholding | Hybrid Rendering Result |
|:-------------------:|:-----------------:|:----------------------------:|:------------------:|:-------------------:|:----------------------:|:-------------:|:----------------------:|:-----------------------------:|:-----------------------:|
| U-Net | 20 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val020/val020_hybrid_render.png" width="250"> |
| MLP | 20 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val020/val020_hybrid_render.png" width="250"> |
| U-Net | 40 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val040/val040_hybrid_render.png" width="250"> |
| MLP | 40 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val040/val040_hybrid_render.png" width="250"> |
| U-Net | 60 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_tunet2/cam_view_val_index_val060/val060_hybrid_render.png" width="250"> |
| MLP | 60 | k = 30 | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_gt.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_gs_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_nerf_pred.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_ssim_error.png" width="255"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_pred_uncertainty.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_mask_threshold.png" width="250"> | <img src="./output_hybrid_threshold0.3_renderall_init15000_simple-mlp2/cam_view_val_index_val060/val060_hybrid_render.png" width="250"> |

Across different camera views, the hybrid rendering results with \(k = 30\) consistently achieve high perceptual quality. Both U-Net and MLP-based uncertainty models effectively identify uncertain regions, allowing the hybrid system to selectively refine key pixels with NeRF. Despite the MLP having significantly fewer parameters, its results remain comparable to those of the U-Net across multiple views. This demonstrates that our uncertainty-aware hybrid framework is robust to changes in viewpoint and that lightweight uncertainty models are sufficient to maintain high rendering fidelity.


---

# Quantitative Results

The following table summarizes the rendering quality and runtime results with Gaussian Splatting method only (GS only), NeRF method only (NeRF only), SS Mip-NeRF method, and our hybrid rendering method at different replacement rates with MLP and U-Net models for uncertainty prediction:

| Method             | PSNR↑ | SSIM↑ | GS Time (s) | NeRF Time (s) | Total Time (s) | Replace % |
|:-------------------|:-----:|:-----:|:-----------:|:-------------:|:--------------:|:---------:|
| GS only            | 22.23 | 0.819 | 0.246       | -             | 0.246          | 0%        |
| NeRF only          | 24.76 | 0.891 | -           | 0.827         | 0.827          | 100%      |
| SS Mip-NeRF        | 32.60 | -     | -           | 0.864         | 0.864          | 100%      |
| Hybrid-MLP 5%      | 22.39 | 0.824 | 0.272       | 0.040         | 0.312          | ~5%       |
| Hybrid-MLP 10%     | 22.64 | 0.832 | 0.272       | 0.081         | 0.353          | ~10%      |
| Hybrid-MLP 30%     | 24.31 | 0.875 | 0.272       | 0.220         | 0.492          | ~30%      |
| Hybrid-MLP 50%     | 24.75 | 0.892 | 0.271       | 0.397         | 0.668          | ~50%      |
| Hybrid-UNet 5%     | 22.58 | 0.836 | 0.275       | 0.040         | 0.315          | ~5%       |
| Hybrid-UNet 10%    | 22.88 | 0.847 | 0.274       | 0.079         | 0.353          | ~10%      |
| Hybrid-UNet 30%    | 24.11 | 0.880 | 0.274       | 0.220         | 0.494          | ~30%      |
| Hybrid-UNet 50%    | 24.73 | 0.892 | 0.274       | 0.399         | 0.673          | ~50%      |

**Note:**  
- GS Time refers to the time spent on feature extraction using the GS model.
- NeRF Time refers to the time spent on ray-marching and rendering using NeRF for selected pixels.

As shown above, our hybrid method achieves higher SSIM scores (up to **0.892**) and comparable PSNR values (up to **24.75**) compared to both the GS-only (PSNR: **22.23**, SSIM: **0.819**) and NeRF-only (PSNR: **24.76**, SSIM: **0.891**) baselines. Notably, the **Hybrid-MLP 50%** model not only matches the perceptual quality of NeRF but also reduces the total rendering time to **0.668s**, significantly faster than NeRF-only (**0.827s**) and the state-of-the-art SS Mip-NeRF (**0.864s**) while maintaining only a modest increase over GS-only (**0.246s**). The **Hybrid-UNet 50%** model has comparable perceptual quality and speed, taking only **0.673s**, only 0.005s more than the **Hybrid-MLP 50%** model, to finish a frame. This demonstrates that our hybrid approach can deliver superior visual fidelity at a fraction of the computational cost.

We further analyze the feature extraction cost with $k=50$ (Replace 50\%) as the following tables, the first one is using the Hybrid-UNet model:


| Steps                          | Time (s) | Percentage |
|-------------------------------|---------:|-----------:|
| whole feature extraction       | 0.2740   | 100%       |
| compute alpha sum              | 0.2180   | 79.56%     |
| compute color variance         | 0.0350   | 12.77%     |
| compute 2D covariance (proj_gau) | 0.0080   | 2.92%      |
| compute 2D covariance area (footprint_area) | 0.0040 | 1.46%      |
| compute view direction         | 0.0034   | 1.24%      |
| model feed forward             | 0.0033   | 1.20%      |
| compute depth and sorting      | 0.0010   | 0.36%      |
| select top-k and mask          | 0.0009   | 0.34%      |
| flatten                        | 0.0004   | 0.15%      |

The second table below shows the profiling of the feature extraction cost with $k=50$ (Replace 50\%) using the Hybrid-MLP model:

| Steps                          | Time (s) | Percentage |
|-------------------------------|---------:|-----------:|
| whole feature extraction       | 0.2710   | 100%       |
| compute alpha sum              | 0.2170   | 80.07%     |
| compute color variance         | 0.0350   | 12.92%     |
| compute 2D covariance (proj_gau) | 0.0080   | 2.95%      |
| compute 2D covariance area (footprint_area) | 0.0040 | 1.48%      |
| compute view direction         | 0.0034   | 1.25%      |
| model feed forward             | 0.0013   | 0.48%      |
| compute depth and sorting      | 0.0010   | 0.37%      |
| select top-k and mask          | 0.0009   | 0.33%      |
| flatten                        | 0.0004   | 0.15%      |


The profiling of the feature extraction process for both the Hybrid-UNet and Hybrid-MLP models shows that most of the time is spent on **alpha sum computation** (∼80%) and **color variance** (∼13%), with all other steps contributing less than 3%.

Hybrid-MLP is slightly faster overall (0.2710s vs. 0.2740s) due to its smaller model and faster inference (0.0013s vs. 0.0033s). Despite this, both models share nearly identical time breakdowns, indicating that the main bottleneck lies in **feature computation** (especially alpha sum), not model complexity.



---

# References

[1] Ben Mildenhall, Pratul P. Srinivasan, Matthew Tancik, Jonathan T. Barron, Ravi Ramamoorthi, and Ren Ng.  
NeRF: Representing scenes as neural radiance fields for view synthesis.  
In *ECCV*, 2020.

[2] Bernhard Kerbl, Georgios Kopanas, Thomas Leimkühler, and George Drettakis.  
3D Gaussian Splatting for Real-Time Radiance Field Rendering.  
*ACM Transactions on Graphics (TOG)*, 42(4), July 2023.

[3] Jonathan T. Barron, Ben Mildenhall, Matthew Tancik, Peter Hedman, Ricardo Martin-Brualla, and Pratul P. Srinivasan.  
Mip-NeRF: A multiscale representation for anti-aliasing neural radiance fields, 2021.